# Join: engine_lab_filtered + project_enriched

Joins `engine_lab_filtered.csv` and `project_enriched.csv` on the five shared key columns:
- **Project ID**
- **Owner Name**
- **Project Name**
- **City**
- **State**

The result is saved to `output/project_joined.csv` and `output/project_joined.xlsx`.

In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path("../")
OUTPUT = ROOT / "output"

filtered  = pd.read_csv(OUTPUT / "engine_lab_filtered.csv")
enriched  = pd.read_csv(OUTPUT / "project_enriched.csv")

print(f"engine_lab_filtered : {filtered.shape[0]:,} rows × {filtered.shape[1]} cols")
print(f"project_enriched    : {enriched.shape[0]:,} rows × {enriched.shape[1]} cols")

engine_lab_filtered : 554 rows × 43 cols
project_enriched    : 554 rows × 23 cols


In [2]:
JOIN_KEYS = ["Project ID", "Owner Name", "Project Name", "City", "State"]

# Verify all join keys exist in both DataFrames
for df_name, df in [("engine_lab_filtered", filtered), ("project_enriched", enriched)]:
    missing = [c for c in JOIN_KEYS if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing join key(s): {missing}")
    print(f"{df_name}: all join keys present ✓")

engine_lab_filtered: all join keys present ✓
project_enriched: all join keys present ✓


In [3]:
# Outer join with indicator to audit match quality before committing to inner join
audit = filtered.merge(enriched, on=JOIN_KEYS, how="outer", indicator=True)
print(audit["_merge"].value_counts().to_string())
print(f"\nTotal outer-join rows : {len(audit):,}")

_merge
both          554
left_only       0
right_only      0

Total outer-join rows : 554


In [4]:
# Inner join — keeps only rows present in both files
joined = filtered.merge(enriched, on=JOIN_KEYS, how="inner")

print(f"Joined dataset : {joined.shape[0]:,} rows × {joined.shape[1]} cols")
joined.head(3)

Joined dataset : 554 rows × 61 cols


,Source File,Industry,Date,Project ID,Umbrella Project ID,Project Name,Owner Name,Plant Name,Umbrella Project Name,TIV (USD),...,baseline_distribution_loss_mwh,high_density_load_share_score,high_density_load_share_explanation,foak_commercial_access_score,foak_commercial_access_explanation,customer_type_label,permitting_burden_index,hv_mv_readiness_gap_score,land_cost_proxy_score,buildable_land_constraint_score
0,Engine Lab 4.pdf,Industrial Manufacturing,20-Jan-2026,301150544,72271.0,ROSEBUD STARGATE DATA CENTER CAMPUS (FREEBIRD)...,SB Energy,Stargate Rosebud Data Center Campus (Freebird),PROJECT STARGATE,1000000000,...,45.618199,8,SB Energy's hyperscale data centers support AI...,7,SB Energy's strategic supply chain leadership ...,enterprise,4.0,6.0,3.0,5.0
1,Engine Lab 3.pdf,Industrial Manufacturing,02-Feb-2026,301083916,72271.0,PORT WASHINGTON DATA CENTER CAMPUS EXP PHASE I...,Vantage Data Centers,Port Washington Data Center Campus (Project Li...,PROJECT STARGATE,1800000000,...,111.374336,10,Vantage's hyperscale campuses support ultra-hi...,7,Vantage's strategic partnerships and leadershi...,hyperscaler,3.0,8.0,7.0,6.0
2,Engine Lab 4.pdf,Industrial Manufacturing,20-Jan-2026,301150841,72271.0,ROSEBUD STARGATE DATA CENTER CAMPUS (FREEBIRD)...,SB Energy,Stargate Rosebud Data Center Campus (Freebird),PROJECT STARGATE,1000000000,...,NaN,8,SB Energy's hyperscale data centers support AI...,7,SB Energy's strategic supply chain leadership ...,enterprise,4.0,6.0,3.0,5.0


In [5]:
# Save outputs
csv_path  = OUTPUT / "project_joined.csv"
xlsx_path = OUTPUT / "project_joined.xlsx"

joined.to_csv(csv_path, index=False)
joined.to_excel(xlsx_path, index=False)

print(f"Saved CSV  → {csv_path}")
print(f"Saved XLSX → {xlsx_path}")

Saved CSV  → ..\output\project_joined.csv
Saved XLSX → ..\output\project_joined.xlsx
